<a href="https://colab.research.google.com/github/Karpagalaksh/Cadetx_DataAnlyst_Project/blob/main/CadetX_Module1_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 — Data Loading & Profiling

## Dataset Source

The initial Week 1 dataset loading approach encountered a Colab/Google Drive
authentication issue. The dataset was therefore sourced through the Kaggle API
using the Lending Club dataset:

`adarshsng/lending-club-loan-data-csv`

## Dataset Scale

- Rows: 2,260,668
- Columns: 145

## Current Progress

- Dataset downloaded using Kaggle API.
- Dataset files verified.
- Dataset dimensions identified.
- Initial memory usage investigated.
- Full DataFrame loading required approximately 5.55 GB of memory.
- Colab RAM limitation identified.
- Memory-efficient chunk-based processing implemented.
- Chunk-based reading tested successfully using 100,000 rows per chunk.
- Data-type profiling is being redesigned for chunk-based processing.
- Missing-value profiling is being redesigned for chunk-based processing.

## Current Challenge

The complete dataset is too large to safely keep in memory as a pandas
DataFrame in the available Colab runtime.

Therefore, the profiling workflow will process the CSV in chunks rather than
loading the complete dataset into memory.

## Current Status

**Module 1 profiling is in progress.**

## Next Steps

- Complete memory-efficient data-type profiling.
- Complete missing-value analysis across the full dataset.
- Perform duplicate analysis.
- Generate the final profiling report.

In [1]:
!pip install -q kaggle

In [2]:
import pandas as pd
import os

DATA_PATH = "lending_club_data2/loan.csv"
CHUNK_SIZE = 100_000

print("Profiling configuration")
print("Data path:", DATA_PATH)
print("Chunk size:", CHUNK_SIZE)

Profiling configuration
Data path: lending_club_data2/loan.csv
Chunk size: 100000


In [3]:
# Data type profiling across the complete dataset

dtype_counts = {}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    for column, dtype in chunk.dtypes.items():
        dtype_name = str(dtype)

        if column not in dtype_counts:
            dtype_counts[column] = dtype_name

print("Column Data Types")
print(pd.Series(dtype_counts).value_counts())

Column Data Types
float64    58
int64      52
object     35
Name: count, dtype: int64


In [4]:
# Download dataset
!kaggle datasets download -d adarshsng/lending-club-loan-data-csv

# Extract without interactive overwrite prompts
!mkdir -p lending_club_data2
!unzip -q -o lending-club-loan-data-csv.zip -d lending_club_data2

print("Dataset downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv
License(s): DbCL-1.0
lending-club-loan-data-csv.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset downloaded and extracted.


In [5]:
!ls -lh lending_club_data2

total 1.2G
-rw-r--r-- 1 root root  24K Jun 17  2021 LCDataDictionary.xlsx
-rw-r--r-- 1 root root 1.2G Jun 17  2021 loan.csv


In [6]:
# Test memory-efficient chunk reading

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    print("First chunk shape:", chunk.shape)
    display(chunk.head())
    break

First chunk shape: (100000, 145)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Data type profiling across the complete dataset

dtype_counts = {}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    for column, dtype in chunk.dtypes.items():
        dtype_name = str(dtype)

        if column not in dtype_counts:
            dtype_counts[column] = dtype_name

print("Column Data Types")
print(pd.Series(dtype_counts).value_counts())

Column Data Types
float64    58
int64      52
object     35
Name: count, dtype: int64


In [10]:
# Missing-value profiling across the complete dataset

missing_counts = None
total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    chunk_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = chunk_missing
    else:
        missing_counts = missing_counts.add(chunk_missing, fill_value=0)

    total_rows += len(chunk)

missing_summary = pd.DataFrame({
    "Missing": missing_counts.astype(int),
    "Missing %": (missing_counts / total_rows * 100).round(4)
})

missing_summary = (
    missing_summary[missing_summary["Missing"] > 0]
    .sort_values("Missing %", ascending=False)
)

print("Columns with missing values:")
display(missing_summary)

Columns with missing values:


,Missing,Missing %
id,2260668,100.0000
member_id,2260668,100.0000
url,2260668,100.0000
orig_projected_additional_accrued_interest,2252242,99.6273
hardship_loan_status,2250055,99.5305
...,...,...
acc_now_delinq,29,0.0013
inq_last_6mths,30,0.0013
delinq_amnt,29,0.0013
annual_inc,4,0.0002


In [11]:
print("Dataset Shape")
print("Rows:", total_rows)
print("Columns:", len(dtype_counts))

Dataset Shape
Rows: 2260668
Columns: 145


In [12]:
# Duplicate analysis
#
# Full-row duplicate detection will be implemented using
# memory-efficient processing so that duplicates across
# different chunks are also detected.

print("Duplicate analysis: pending memory-efficient implementation.")

Duplicate analysis: pending memory-efficient implementation.


In [12]:
dup_count = df.duplicated().sum()
dup_pct = round(dup_count/len(df)*100,2)
print("Duplicate_Rows: ", dup_count)
print("Duplicate_%: ", dup_pct)

Duplicate_Rows:  0
Duplicate_%:  0.0


In [13]:
profiling_report = {
    "dataset": {
        "rows": int(total_rows),
        "columns": int(len(dtype_counts))
    },
    "data_types": {
        dtype: int(count)
        for dtype, count in pd.Series(dtype_counts).value_counts().items()
    },
    "missing_values": {
        "columns_with_missing_values": int(len(missing_summary)),
        "columns_with_100_percent_missing": int(
            (missing_summary["Missing %"] == 100).sum()
        )
    },
    "duplicates": {
        "status": "Pending memory-efficient analysis"
    }
}

profiling_report

{'dataset': {'rows': 2260668, 'columns': 145},
 'data_types': {'float64': 58, 'int64': 52, 'object': 35},
 'missing_values': {'columns_with_missing_values': 113,
  'columns_with_100_percent_missing': 3},
 'duplicates': {'status': 'Pending memory-efficient analysis'}}